# 01c · Apply ERP-derived epoch QC to High-Gamma

ERP/HG 使用完全相同的 trial keep mask，避免模态间 trial 不一致。

In [ ]:
# [Apply] Verify trigger alignment before filtering
from pathlib import Path
import sys, numpy as np
ROOT=Path('/home/lirui/liulab_project/ieeg/Project_colorieeg_2026'); PIPE=ROOT/'color_cognition_pipeline'/'analyse_0720'
sys.path.insert(0,str(PIPE)); import config
from utils.epochs import load_epochs, save_epochs
for task in config.RUNS:
    path=config.INTERMEDIATE_ROOT/'test001'/'preprocessing'/f'task{task}_hg.npz'; hg=load_epochs(path,prefer_clean=False)
    qc=np.load(config.subject_result_dir('test001')/'preprocessing'/f'task{task}_epoch_qc.npz',allow_pickle=False); keep=qc['keep']
    if len(keep)!=len(hg['triggers']) or not np.array_equal(qc['trigger'].astype(str),hg['triggers'].astype(str)): raise RuntimeError(f'Task {task}: ERP/HG trial alignment failed')
    save_epochs(path.with_name(f'task{task}_hg_clean.npz'),hg['data'][keep],hg['times_ms'],hg['triggers'][keep],hg['channel_names'],{'task':task,'epoch_qc_source':str(config.subject_result_dir('test001')/'preprocessing'/f'task{task}_epoch_qc.npz'),'rejected_epochs':int((~keep).sum())})
print('HG trial QC synchronized')